In [2]:
import os
import pandas as pd
import plotly.graph_objects as go
import pickle

from random import sample

from lidar_object_tracking.config import PROCESSED_DATA_DIR
from lidar_object_tracking.dataset import main



In [9]:
data_directory = '/home/danielb/Desktop/CodingProjects/lidar_object_tracking/data/raw/'

from lidar_object_tracking.dataset import unpack_dataset

frame_dict = unpack_dataset(data_directory, '004')

/home/danielb/Desktop/CodingProjects/lidar_object_tracking/lidar_object_tracking/dataset.py:48: VisibleDeprecationWarning:

dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)



In [ ]:
import math

def rotate_45(x_val, y_val):
    angle = -math.pi/4
    x_rot = x_val*math.cos(angle) + y_val*math.sin(angle)
    y_rot = -x_val*math.sin(angle) + y_val*math.cos(angle)
    return x_rot,y_rot


for i in frame_dict.keys():
    frame_dict[i]['xr'],frame_dict[i]['yr'] = (
        rotate_45(frame_dict[i]['x'],frame_dict[i]['y'])[0],
        rotate_45(frame_dict[i]['x'],frame_dict[i]['y'])[1]
        )

for i in frame_dict.keys():
    frame_dict[i].drop(frame_dict[i][(frame_dict[i]['xr'].apply(abs)>25)|
                                     (frame_dict[i]['yr']<0)|
                                     (frame_dict[i]['yr'] >25)|
                                     (frame_dict[i]['z']>5)|
                                     (frame_dict[i]['z']<0)].index, inplace = True)

# for i in frame_dict.keys():
#    frame_dict[i]['z_zero'] = frame_dict[i]['z'].apply(lambda z: z*0)


In [ ]:
sub_sample = [sample(list(frame_dict[i].index), 1500) for i in range(len(frame_dict))]

coord_by_frame = {i:frame_dict[i][['xr','yr','z']].loc[sub_sample[i]] for i in range(len(frame_dict))}

coord_by_frame

In [ ]:
fig = go.Figure()

for i in range(80):
    fig.add_trace(
        go.Scatter3d(
            visible = False,
            mode = 'markers',
            marker = dict(size = 2),
            x = coord_by_frame[i]['xr'],
            y = coord_by_frame[i]['yr'],
            z = coord_by_frame[i]['z']
        )
    )
    fig.update_layout(height = 500, width = 750)
    fig.update_scenes(xaxis =dict(range = [-25,25]), 
                      yaxis=dict(range = [0,25]), 
                      zaxis=dict(range = [0,5]),
                      aspectmode = 'data')

fig.data[0].visible = True

steps = []
for i in range(len(list(fig.data))):
    step = dict(
        method = 'update',
        args= [{'visible':[False]*len(list(fig.data))}],
        label = f'{i/10}s',
    )
    step['args'][0]['visible'][i] = True
    steps.append(step)

sliders = [dict(
    active = 0,
    currentvalue = {'prefix': "Frame: ",},
    pad={'t':80},
    steps = steps
)]

fig.update_layout(
    sliders = sliders
)

fig.show()

In [ ]:
from lidar_object_tracking.dataset import data_to_json

data_to_json(coord_by_frame, PROCESSED_DATA_DIR, scene_number = '004')

In [ ]:
from lidar_object_tracking.dataset import read_data_json

raw_json = read_data_json(PROCESSED_DATA_DIR, '004')

In [ ]:
df_dataset = {}
for key in raw_json.keys():
    df_dataset[int(key)] = pd.DataFrame(raw_json[key],columns = ['xr','yr','z'])

df_dataset

In [ ]:
from lidar_object_tracking.plots import plot_with_slider
plot_with_slider(coord_by_frame)

In [7]:
from lidar_object_tracking.plots import plot_with_slider
plot_with_slider(df_dataset)

In [5]:
from lidar_object_tracking.dataset import load_json_data

df_dataset = load_json_data(data_directory=PROCESSED_DATA_DIR, scene_num='004')

In [1]:
from lidar_object_tracking.dataset import unpack_to_json

unpack_to_json()

2026-02-20 12:59:37.820 | INFO     | lidar_object_tracking.config:<module>:11 - PROJ_ROOT path is: /home/danielb/Desktop/CodingProjects/lidar_object_tracking


/home/danielb/Desktop/CodingProjects/lidar_object_tracking/lidar_object_tracking/dataset.py:44: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  frame_dict[int(dir.name[0:2])] = pd.DataFrame(pickle.load(file))
